# 06 - オブジェクト分析

ドラゴン、バロン、リフトヘラルドの獲得と勝率の関係を分析する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import yaml
from pathlib import Path

for font in ['Meiryo', 'Yu Gothic', 'MS Gothic', 'Hiragino Sans']:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        plt.rcParams['font.family'] = font
        break
    except ValueError:
        continue
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])

DATA = Path('..') / 'data' / 'processed'

df_obj = pd.read_csv(DATA / 'objectives.csv')
df_matches = pd.read_csv(DATA / 'matches.csv', parse_dates=['gameCreation'])
df_players = pd.read_csv(DATA / 'player_stats.csv')

with open(Path('..') / 'config' / 'settings.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
MEMBERS = {f'{m["game_name"]}#{m["tag_line"]}' for m in cfg['members']}

df_players['riotId'] = df_players['summonerName'] + '#' + df_players['tagLine']
member_teams = df_players[df_players['riotId'].isin(MEMBERS)][['matchId', 'teamId']].drop_duplicates()

print(f'オブジェクトイベント数: {len(df_obj)}')
print(f'試合数: {len(df_matches)}')

In [ ]:
# ドラゴン獲得数 vs 勝率
dragons = df_obj[df_obj['objectiveType'] == 'DRAGON'].copy()

dragon_counts = dragons.groupby(['matchId', 'teamId']).size().reset_index(name='dragonCount')

dragon_with_win = dragon_counts.merge(member_teams, on=['matchId', 'teamId'], how='inner')
dragon_with_win = dragon_with_win.merge(
    df_matches[['matchId', 'team100Win']], on='matchId', how='left'
)
dragon_with_win['win'] = (
    (dragon_with_win['teamId'] == 100) & dragon_with_win['team100Win']
) | (
    (dragon_with_win['teamId'] == 200) & ~dragon_with_win['team100Win']
)

dragon_wr = dragon_with_win.groupby('dragonCount').agg(
    games=('win', 'count'),
    winrate=('win', 'mean'),
).reset_index()
dragon_wr['winrate_pct'] = (dragon_wr['winrate'] * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if wr < 50 else '#2ecc71' for wr in dragon_wr['winrate_pct']]
bars = ax.bar(dragon_wr['dragonCount'], dragon_wr['winrate_pct'], color=colors)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('ドラゴン獲得数')
ax.set_ylabel('勝率 %')
ax.set_title('ドラゴン獲得数と勝率の関係')

for bar, n in zip(bars, dragon_wr['games']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'n={n}',
            ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ドラゴン種別ごとの勝率影響
dragon_types = dragons.merge(member_teams, on=['matchId', 'teamId'], how='inner')
dragon_types = dragon_types.merge(
    df_matches[['matchId', 'team100Win']], on='matchId', how='left'
)
dragon_types['win'] = (
    (dragon_types['teamId'] == 100) & dragon_types['team100Win']
) | (
    (dragon_types['teamId'] == 200) & ~dragon_types['team100Win']
)

dtype_wr = dragon_types.groupby('monsterSubType').agg(
    獲得数=('win', 'count'),
    勝率=('win', 'mean'),
).round(2)
dtype_wr['勝率%'] = (dtype_wr['勝率'] * 100).round(1)
dtype_wr.sort_values('勝率', ascending=False)

In [ ]:
# ファーストドラゴン / ファーストバロンの勝率インパクト
for obj_type, title in [('DRAGON', 'ファーストドラゴン'), ('BARON', 'ファーストバロン')]:
    firsts = df_obj[(df_obj['objectiveType'] == obj_type) & (df_obj['isFirst'] == True)].copy()
    firsts = firsts.merge(member_teams, on=['matchId', 'teamId'], how='inner')
    firsts = firsts.merge(df_matches[['matchId', 'team100Win']], on='matchId', how='left')
    firsts['win'] = (
        (firsts['teamId'] == 100) & firsts['team100Win']
    ) | (
        (firsts['teamId'] == 200) & ~firsts['team100Win']
    )

    if len(firsts) > 0:
        wr_taken = firsts['win'].mean() * 100
        print(f'{title}を取得した場合の勝率: {wr_taken:.1f}% ({len(firsts)}試合)')
    else:
        print(f'{title}: データなし')

In [ ]:
# バロン獲得タイミングと勝率
barons = df_obj[df_obj['objectiveType'] == 'BARON'].copy()
barons = barons.merge(member_teams, on=['matchId', 'teamId'], how='inner')
barons = barons.merge(df_matches[['matchId', 'team100Win']], on='matchId', how='left')
barons['win'] = (
    (barons['teamId'] == 100) & barons['team100Win']
) | (
    (barons['teamId'] == 200) & ~barons['team100Win']
)

if not barons.empty:
    baron_bins = [20, 25, 30, 35, 40, 45, 60]
    baron_labels = ['20-25', '25-30', '30-35', '35-40', '40-45', '45+']
    barons['timeBin'] = pd.cut(barons['timestampMin'], bins=baron_bins, labels=baron_labels)

    baron_wr = barons.groupby('timeBin', observed=False).agg(
        games=('win', 'count'),
        winrate=('win', 'mean'),
    ).reset_index()
    baron_wr['winrate_pct'] = (baron_wr['winrate'] * 100).round(1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(baron_wr['timeBin'].astype(str), baron_wr['winrate_pct'], color='#9b59b6')
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('バロン獲得タイミング（分）')
    ax.set_ylabel('勝率 %')
    ax.set_title('バロン獲得タイミング別 勝率')
    plt.tight_layout()
    plt.show()
else:
    print('バロンデータなし')

In [ ]:
# リフトヘラルド → タワー破壊の連鎖分析
heralds = df_obj[
    (df_obj['objectiveType'] == 'RIFT_HERALD')
].merge(member_teams, on=['matchId', 'teamId'], how='inner')

towers = df_obj[df_obj['objectiveType'] == 'TOWER'].copy()

herald_tower = []
for _, herald in heralds.iterrows():
    mid = herald['matchId']
    tid = herald['teamId']
    h_time = herald['timestampMin']
    towers_after = towers[
        (towers['matchId'] == mid) &
        (towers['teamId'] == tid) &
        (towers['timestampMin'] > h_time) &
        (towers['timestampMin'] <= h_time + 3)
    ]
    herald_tower.append({
        'matchId': mid,
        'heraldTime': h_time,
        'towersWithin3min': len(towers_after),
    })

df_ht = pd.DataFrame(herald_tower)
if not df_ht.empty:
    print('リフトヘラルド取得後3分以内のタワー破壊数:')
    print(df_ht['towersWithin3min'].value_counts().sort_index())
    print(f'\n平均: {df_ht["towersWithin3min"].mean():.2f}本')
else:
    print('ヘラルドデータなし')